# delbina-sum: teacher run

Generates dataset v0, summarizes it with the teacher (Qwen2.5-7B-Instruct-AWQ), keeps only summaries
that pass the fact check, and pushes the result to the private HF dataset `delbina/delbina-sum-data`.

**Runtime**: Colab → Runtime ▸ Change runtime type ▸ T4 GPU. Kaggle → Accelerator GPU T4 x2, Internet on.
Both need a secret named `HF_TOKEN` (Colab: key icon; Kaggle: Add-ons ▸ Secrets).

**Pilot first**: `PILOT = 100` runs a few minutes on Colab and pushes nothing. Read the samples it prints.
Then set `PILOT = 0` and run the full set (best on Kaggle with Save Version ▸ Save & Run All).

In [ ]:
# --- settings ---
PILOT = 100                     # 0 = full run (train + val + test) and push to HF
TEACHER = "Qwen/Qwen2.5-7B-Instruct-AWQ"
MAX_NEW_TOKENS = 768            # teacher summary cap; truncated summaries are rejected
MAX_MODEL_LEN = 6144            # longest v0 prompt is ~4k tokens
HF_REPO = "delbina/delbina-sum-data"
VERSION = "v0"

In [ ]:
# --- platform + token (works on Kaggle and Colab) ---
import os
if os.environ.get("KAGGLE_KERNEL_RUN_TYPE"):          # set only inside Kaggle notebooks
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    PLATFORM, WORK = "kaggle", "/kaggle/working"
else:
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
    PLATFORM, WORK = "colab", "/content"
print(PLATFORM, "token loaded:", bool(os.environ["HF_TOKEN"]))

In [ ]:
# --- install (3-6 min; vLLM brings its own torch) ---
!pip install -q vllm "delbina==0.4.0" huggingface_hub
!rm -rf {WORK}/delbina-sum && git clone -q https://github.com/delbina/delbina-sum.git {WORK}/delbina-sum
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# --- generate v0 (deterministic, identical to running it on your laptop) ---
import sys, json
sys.path.insert(0, f"{WORK}/delbina-sum/datagen")
import make_dataset as md
from common import clean_summary, score, student_prompt, teacher_prompt

sizes = {"train": PILOT} if PILOT else {"train": 2000, "val": 200, "test": 300}
data = {split: [md.make_sample(split, i) for i in range(n)] for split, n in sizes.items()}
print({k: len(v) for k, v in data.items()})

In [ ]:
# --- load teacher ---
import os, sys
# vLLM's warmup imports `cutlass`, which its package exposes via a .pth file that Python only
# runs at startup. This kernel started before the install, so add the path by hand (no restart).
import nvidia_cutlass_dsl
_dsl = os.path.join(nvidia_cutlass_dsl.__path__[0], "dsl_packages")
sys.path.insert(0, _dsl)
os.environ["PYTHONPATH"] = _dsl + os.pathsep + os.environ.get("PYTHONPATH", "")  # for the engine process
import cutlass  # fail here, clearly, not inside the engine

import torch
from vllm import LLM, SamplingParams

llm = LLM(model=TEACHER, dtype="half", tensor_parallel_size=torch.cuda.device_count(),
          max_model_len=MAX_MODEL_LEN, gpu_memory_utilization=0.90, seed=0)
params = SamplingParams(temperature=0.0, max_tokens=MAX_NEW_TOKENS)

In [ ]:
# --- summarize ---
import time
for split, rows in data.items():
    t = time.time()
    convs = [[{"role": "user", "content": teacher_prompt(r["lines"])}] for r in rows]
    outs = llm.chat(convs, params, use_tqdm=True)
    for r, o in zip(rows, outs):
        r["summary"] = clean_summary(o.outputs[0].text)
        r["truncated"] = o.outputs[0].finish_reason == "length"
        r["score"] = score(r["summary"], r["facts"])
        r["keep"] = r["score"]["ok"] and not r["truncated"]
    print(f"{split}: {len(rows)} in {time.time() - t:.0f}s")

In [ ]:
# --- stats ---
from collections import defaultdict
def stats(rows):
    n = len(rows)
    by_kind, by_where = defaultdict(list), defaultdict(list)
    for r in rows:
        missing = set(r["score"]["missing"])
        for f in r["facts"]:
            hit = f["value"] not in missing
            by_kind[f["kind"]].append(hit); by_where[f["where"]].append(hit)
    lens = sorted(len(r["summary"]) for r in rows)
    pct = lambda xs: round(100 * sum(xs) / len(xs), 1) if xs else None
    return {
        "samples": n, "kept": sum(r["keep"] for r in rows),
        "keep_rate": pct([r["keep"] for r in rows]),
        "fact_recall": pct([h for v in by_kind.values() for h in v]),
        "stale_hit_rate": pct([bool(r["score"]["stale_hits"]) for r in rows]),
        "truncated": sum(r["truncated"] for r in rows),
        "recall_by_kind": {k: pct(v) for k, v in sorted(by_kind.items())},
        "recall_by_where": {k: pct(v) for k, v in sorted(by_where.items())},
        "summary_chars_p50": lens[n // 2], "summary_chars_p95": lens[int(n * 0.95)],
    }
STATS = {"teacher": TEACHER, "version": VERSION, **{s: stats(r) for s, r in data.items()}}
print(json.dumps(STATS, indent=1))

In [ ]:
# --- read 10 by eye: 5 kept, 5 rejected ---
rows = data["train"]
show = [r for r in rows if r["keep"]][:5] + [r for r in rows if not r["keep"]][:5]
for r in show:
    print("=" * 100)
    print(r["id"], "KEEP" if r["keep"] else "REJECT", "| facts:", [(f["subject"], f["value"], f["stale"]) for f in r["facts"]])
    print("missing:", r["score"]["missing"], "| stale:", r["score"]["stale_hits"], "| truncated:", r["truncated"])
    print("-" * 100)
    print(r["summary"])

In [ ]:
# --- save + push (full run only) ---
from pathlib import Path
from huggingface_hub import HfApi

out = Path(WORK) / "out" / VERSION
out.mkdir(parents=True, exist_ok=True)
def write(name, rows):
    with open(out / name, "w", encoding="utf-8") as fh:
        for r in rows:
            fh.write(json.dumps({**r, "prompt": student_prompt(r["lines"]), "teacher": TEACHER}, ensure_ascii=False) + "\n")

for split, rows in data.items():
    if split == "test":
        write("test.jsonl", rows)                      # all kept: teacher summaries are a reference score only
    else:
        write(f"{split}.jsonl", [r for r in rows if r["keep"]])
        write(f"{split}_rejected.jsonl", [r for r in rows if not r["keep"]])
(out / "stats.json").write_text(json.dumps(STATS, indent=1))
print(sorted(p.name for p in out.iterdir()))

if PILOT:
    print("pilot: nothing pushed")
else:
    api = HfApi()
    api.create_repo(HF_REPO, repo_type="dataset", private=True, exist_ok=True)
    api.upload_folder(folder_path=str(out), path_in_repo=VERSION, repo_id=HF_REPO, repo_type="dataset",
                      commit_message=f"{VERSION}: teacher {TEACHER}")
    print(f"pushed to https://huggingface.co/datasets/{HF_REPO}/tree/main/{VERSION}")